## Chat with your PDF files using LlamaIndex, Astra DB (Apache Cassandra), and Gradient's open-source models, including LLama2 and Streamlit.

Click the link below to open this notebook in Colab.

<a href="https://colab.research.google.com/github/Ankit-thepe/pdf-q-a-llamaindex-llama2/blob/main/pdf-q-a-notebook.ipynb" target="_blank"><img height="40" alt="Run your own notebook in Colab" src="https://colab.research.google.com/assets/colab-badge.svg"></a>

# Installation

In [ ]:
!pip install -q cassandra-driver
!pip install -q cassio>=0.1.1
!pip install -q gradientai --upgrade
!pip install -q llama-index
!pip install -q pypdf
!pip install -q tiktoken==0.4.0

# Set Environment Variables

In [ ]:
import os
import json

os.environ['GRADIENT_ACCESS_TOKEN'] = 'Enter your GRADIENT ACCESS TOKEN'
os.environ['GRADIENT_WORKSPACE_ID'] = 'Enter your GRADIENT WORKSPACE ID'

# Import Cassandra & LlamaIndex

In [ ]:
from cassandra.auth import PlainTextAuthProvider
from cassandra.cluster import Cluster
from llama_index import ServiceContext, set_global_service_context
from llama_index import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.embeddings import GradientEmbedding
from llama_index.llms import GradientBaseModelLLM
from llama_index.vector_stores import CassandraVectorStore

# Connect to Astra DB (VectorDB)

In [ ]:
# Update file names below to match your downloaded Astra credentials
cloud_config = {'secure_connect_bundle': 'secure-connect-ankit-astra-test.zip'}

with open('ankit_astra_test-token.json') as f:
    secrets = json.load(f)

CLIENT_ID = secrets['clientId']
CLIENT_SECRET = secrets['secret']

auth_provider = PlainTextAuthProvider(CLIENT_ID, CLIENT_SECRET)
cluster = Cluster(cloud=cloud_config, auth_provider=auth_provider)
session = cluster.connect()

row = session.execute('select release_version from system.local').one()
print(row[0] if row else 'Connection error')

# Configure LLM (Llama2)

In [ ]:
llm = GradientBaseModelLLM(base_model_slug='llama2-7b-chat', max_tokens=400)

# Configure Embeddings

In [ ]:
embed_model = GradientEmbedding(
    gradient_access_token=os.environ['GRADIENT_ACCESS_TOKEN'],
    gradient_workspace_id=os.environ['GRADIENT_WORKSPACE_ID'],
    gradient_model_slug='bge-large')

service_context = ServiceContext.from_defaults(llm=llm, embed_model=embed_model, chunk_size=256)
set_global_service_context(service_context)

# Load PDFs & Query

In [ ]:
documents = SimpleDirectoryReader('/content/Documents').load_data()
print(f'Loaded {len(documents)} document(s).')

In [ ]:
index = VectorStoreIndex.from_documents(documents, service_context=service_context)
query_engine = index.as_query_engine()

In [ ]:
response = query_engine.query('What is Positional Encoding?')
print(response)